# Разработка алгоритма для сентимент анализа отзывов к медицинским учреждениям

## Сентимент анализ собранного датасета

### Уставка и подключение зависимостей

In [ ]:
!pip install pyabsa

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import BertTokenizerFast

from pyabsa import AspectTermExtraction as ATEPC

import pandas as pd

import warnings
warnings.filterwarnings("ignore")

### Анализ тональности отзывов с использованием модели BERT

Этот этап предназначен для анализа тональности текстов на русском языке с помощью модели `blanchefort/rubert-base-cased-sentiment`, которая обучена для классификации текстов в три категории: **положительная**, **нейтральная** и **отрицательная**.

**Описание шагов**:

1. **Токенизация текста**: Входной текст преобразуется в токены, которые затем преобразуются в числовые данные для подачи в модель.
2. **Прогон через модель**: Модель на основе BERT анализирует текст и генерирует "логиты", которые представляют собой сырые прогнозы для каждой категории.
3. **Применение Softmax**: Функция softmax применяется к логитам, чтобы преобразовать их в вероятности для каждой категории.
4. **Выбор категории**: Модель выбирает категорию с наибольшей вероятностью и возвращает соответствующую тональность.
5. **Результат**: Функция возвращает одну из следующих тональностей:
   - **Neutral** (Нейтральный)
   - **Positive** (Положительный)
   - **Negative** (Отрицательный)


In [ ]:
# Загружаем токенизатор и модель для классификации тональности
# Модель 'blanchefort/rubert-base-cased-sentiment' — это предварительно обученная модель для анализа тональности текстов на русском языке
tokenizer = BertTokenizerFast.from_pretrained('blanchefort/rubert-base-cased-sentiment')
model = AutoModelForSequenceClassification.from_pretrained('blanchefort/rubert-base-cased-sentiment', return_dict=True)

# Определяем функцию для предсказания тональности отзыва
@torch.no_grad()
def sentiment_predict(text):
    # Токенизируем входной текст: 
    # - max_length=512: максимальная длина текста (обрезка/дополнение)
    # - padding=True: дополнение текста до max_length
    # - truncation=True: обрезка текста до max_length, если он длиннее
    # - return_tensors='pt': возвращаем данные в формате, подходящем для PyTorch
    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')
    
    # Прогоняем токенизированный текст через модель для получения логитов (выходных данных)
    # Логиты — это сырые значения, которые модель использует для принятия решения
    outputs = model(**inputs)
    
    # Применяем softmax для получения вероятностей для каждого из классов (тональность)
    predicted = torch.nn.functional.softmax(outputs.logits, dim=1)
    
    # Выбираем класс с максимальной вероятностью, что соответствует предсказанному настроению
    predicted = torch.argmax(predicted, dim=1).numpy()
    
    # В зависимости от предсказанного класса, определяем тональность отзыва
    if predicted == 0:
        overall_sentiment = 'Neutral'  # Нейтральный отзыв
    elif predicted == 1:
        overall_sentiment = 'Positive'  # Положительный отзыв
    else:
        overall_sentiment = 'Negative'  # Отрицательный отзыв
    
    # Возвращаем результат
    return overall_sentiment


### Анализ аспектов тональности с использованием модели ATEPC

Этот этап предназначен для извлечения аспектов из текста с использованием модели ATEPC (Aspect Term Extraction and Polarity Classification), обученной для многоклассовой классификации аспектов и их полярности.

**Описание шагов**:

1. **Создание экземпляра модели ATEPC**: Мы инициализируем `aspect_extractor`, который использует предобученную модель для извлечения аспектов и определения их тональности (положительная, нейтральная или отрицательная).
   
2. **Анализ текста**: 
    - Входной текст передается в модель, и результат анализа включает аспекты, их тональность и уверенность модели в каждом аспекте.
    - Для каждого аспекта выводится строка в формате: `аспект: тональность (уверенность)`, где уверенность округляется до 3 знаков после запятой.

3. **Обработка исключений**: В случае возникновения ошибки (например, если модель не может обработать текст), возвращается сообщение об ошибке.

4. **Возвращение результата**: Если аспекты найдены, возвращается строка с перечислением аспектов в виде JSON-подобного формата. Если аспекты не найдены, функция возвращает `None`.


In [ ]:
aspect_extractor = ATEPC.AspectExtractor('multilingual')

def aspect_analysis(text):
    try:
        atepc_result = aspect_extractor.predict(text, print_result=False, save_result=False)
        aspects = atepc_result['aspect']
        sentiments = atepc_result['sentiment']
        confidences = atepc_result['confidence']

        if not aspects:
            return None

        # Возвращаем список аспектов в виде строки JSON-подобного формата
        return "; ".join([
            f"{asp}: {sent} (увер. {round(conf, 3)})"
            for asp, sent, conf in zip(aspects, sentiments, confidences)
        ])
    except Exception as e:
        return f"Ошибка анализа: {e}"


### Обработка исходных данных

In [ ]:
df = pd.read_excel('datasets/reviews_1098.xlsx')

df['общая_тональность'] = df['review_text'].apply(sentiment_predict)
df['аспектный_анализ'] = df['review_text'].apply(aspect_analysis)

df.to_excel('datasets/reviews_with_sentiment_aspects.xlsx', index=False)